# Detekcija AI-generisanog teksta
**Dimitrije Ignjić — RA129/2022**  
Osnove računarske inteligencije, 2025/2026

---

Zadatak je **binarna klasifikacija**: za dati tekst odrediti da li ga je napisao čovjek (klasa `0`) ili ga je generisala AI (klasa `1`).  
Dataset: [AI vs Human Text Dataset](https://www.kaggle.com/datasets/shanegerami/ai-vs-human-text/data) (~487 000 instanci).

Implementiramo i poredimo tri modela obrađena na predmetu: **KNN**, **MLP** i **CNN (1D)**.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

from config import *
from data import load_dataset, sample_balanced, preprocess_text, plot_class_distribution
from features import (
    build_numerical_matrix, plot_numerical_features,
    build_tfidf, train_word2vec, build_word2vec_assets, texts_to_sequences
)
from models import MLP, TextCNN
from train import train_model, predict, make_loader, plot_loss
from evaluate import print_and_plot_metrics, plot_comparison, kfold_knn, kfold_torch

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Uređaj: {device}')

## 1. Učitavanje i eksploracija podataka

Dataset sadrži ~487 000 tekstova (članci, eseji, odgovori na pitanja). Svaki tekst je označen oznakom `0` (čovjek) ili `1` (AI). Zbog neizbansiranosti klasa (305k vs 181k), koristimo **stratifikovano uzorkovanje** — uzimamo jednako instanci iz svake klase.

In [ ]:
df = load_dataset('AI_Human.csv')
df_sample = sample_balanced(df, N_SAMPLE)
plot_class_distribution(df_sample, 'plots/01_raspodjela_klasa.png')
plt.figure(figsize=(5,4))
df_sample['generated'].value_counts().sort_index().plot(kind='bar', color=['steelblue','salmon'])
plt.xticks([0,1], ['Čovjek (0)', 'AI (1)'], rotation=0)
plt.title('Raspodjela klasa u uzorku')
plt.ylabel('Broj instanci')
plt.tight_layout()
plt.show()

Uzorak je savršeno izbalansiran: 30 000 primjera po klasi. Ovo je važno jer neuravnotežen skup može iskriviti metrike (posebno accuracy).

## 2. Ekstrakcija obeležja

### 2.1 Numerička obeležja

Računamo 6 obeležja koja ne zahtijevaju specijalne NLP biblioteke:

| Obeležje | Opis |
|---|---|
| `avg_sentence_len` | Prosječan broj riječi u rečenici |
| `sentence_len_std` | Standardna devijacija dužine rečenica |
| `avg_word_len` | Prosječan broj karaktera po riječi |
| `ttr` | Type-Token Ratio — bogatstvo rječnika |
| `punct_freq` | Udio interpunkcijskih znakova u tekstu |
| `num_sentences` | Ukupan broj rečenica |

In [ ]:
labels = df_sample['generated'].values.astype(int)
X_num = build_numerical_matrix(df_sample['text'])

In [ ]:
plot_numerical_features(X_num, labels, 'plots/02_numericka_obelezja.png')

import pandas as pd
import seaborn as sns
cols = ['avg_sentence_len','sentence_len_std','avg_word_len','ttr','punct_freq','num_sentences']
df_plot = pd.DataFrame(X_num, columns=cols)
df_plot['klasa'] = labels

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, feat in zip(axes, cols[:5]):
    for cls, label, color in [(0,'Čovjek','steelblue'), (1,'AI','salmon')]:
        ax.hist(df_plot[df_plot['klasa']==cls][feat], bins=40, alpha=0.6,
                label=label, color=color, density=True)
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)
plt.suptitle('Distribucija numeričkih obeležja po klasama')
plt.tight_layout()
plt.show()

Vidljive su razlike između klasa, posebno kod `ttr` (AI tekst ima niže bogatstvo rječnika) i `avg_sentence_len` (AI piše duže rečenice). Ova obeležja sama po sebi nisu dovoljno diskriminativna, ali dopunjuju TF-IDF u KNN modelu.

### 2.2 TF-IDF vektorizacija

TF-IDF (Term Frequency–Inverse Document Frequency) mjeri važnost svake riječi relativno prema cijelom korpusu. Koristimo 10 000 najvažnijih obeležja. Ova reprezentacija zanemaruje redosled riječi (bag-of-words), ali je izuzetno efikasna za klasifikaciju teksta.

In [ ]:
texts_clean = df_sample['text'].apply(preprocess_text).tolist()
X_tfidf, tfidf_vectorizer = build_tfidf(texts_clean)

### 2.3 Word2Vec embeddings

Word2Vec treniramo na trening skupu. Svaka riječ se reprezentuje kao vektor od 100 realnih brojeva, pri čemu su semantički slične riječi bliske u tom prostoru. Ovi vektori koriste se kao ulaz u CNN — svaki tekst postaje matrica dimenzija `200 × 100` (max 200 riječi, svaka kao 100-dimenzionalni vektor).

In [ ]:
tokenized = [t.split() for t in texts_clean]
w2v = train_word2vec(tokenized)
word2idx, embedding_matrix = build_word2vec_assets(w2v)
X_sequences = texts_to_sequences(tokenized, word2idx)

# Primjer: vektori semantički sličnih riječi su bliski
print('Najsličnije riječi uz "essay":')
if 'essay' in w2v.wv:
    for word, score in w2v.wv.most_similar('essay', topn=5):
        print(f'  {word:20s} {score:.4f}')

## 3. Podjela na trening i test skup (80/20)

Isti split koristimo za sva tri modela — to je uslov za fer poređenje.

In [ ]:
idx = np.arange(len(labels))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=RANDOM_STATE, stratify=labels)

X_num_train,   X_num_test   = X_num[idx_train],        X_num[idx_test]
X_tfidf_train, X_tfidf_test = X_tfidf[idx_train],      X_tfidf[idx_test]
X_seq_train,   X_seq_test   = X_sequences[idx_train],  X_sequences[idx_test]
y_train, y_test              = labels[idx_train],       labels[idx_test]

print(f'Trening: {len(idx_train)} instanci')
print(f'Test:    {len(idx_test)} instanci')

results = {}

## 4. Model 1 — KNN

KNN klasifikuje novu instancu na osnovu klase K najbližih primjera iz trening skupa. Budući da KNN loše radi u visokim dimenzijama (*kletva dimenzionalnosti*), TF-IDF vektore (10 000 dim) reduciramo pomoću **TruncatedSVD** na 100 dimenzija, te ih kombinujemo s numeričkim obeležjima.

Optimalni K biramo 3-fold unakrsnom validacijom.

In [ ]:
svd = TruncatedSVD(n_components=N_SVD, random_state=RANDOM_STATE)
X_svd_train = svd.fit_transform(X_tfidf_train)
X_svd_test  = svd.transform(X_tfidf_test)

scaler_num = StandardScaler()
X_knn_train = np.hstack([X_svd_train, scaler_num.fit_transform(X_num_train)])
X_knn_test  = np.hstack([X_svd_test,  scaler_num.transform(X_num_test)])

# Odabir K
k_values = [3, 5, 7, 11, 15]
k_scores = []
small_idx = np.random.choice(len(X_knn_train), size=10_000, replace=False)

for k in k_values:
    score = cross_val_score(
        KNeighborsClassifier(n_neighbors=k, metric='euclidean', n_jobs=-1),
        X_knn_train[small_idx], y_train[small_idx], cv=3, scoring='f1_weighted'
    ).mean()
    k_scores.append(score)
    print(f'K={k:2d} → F1={score:.4f}')

best_k = k_values[int(np.argmax(k_scores))]
print(f'\nNajbolji K: {best_k}')

plt.figure(figsize=(6, 4))
plt.plot(k_values, k_scores, '-o', color='steelblue')
plt.xlabel('K')
plt.ylabel('F1-score (3-fold CV)')
plt.title('KNN — Odabir optimalnog K')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean', n_jobs=-1)
knn.fit(X_knn_train, y_train)
y_pred_knn = knn.predict(X_knn_test)
results['KNN'] = print_and_plot_metrics(y_test, y_pred_knn, 'KNN', 'plots/cm_knn.png')

KNN postiže ~95.7% tačnosti. Glavno ograničenje je brzina — pretraga svih 48 000 trening primjera za svaku predikciju je spora.

## 5. Model 2 — MLP (Višeslojna neuronska mreža)

MLP prima TF-IDF vektor (10 000 dimenzija) i prolazi kroz dva skrivena sloja sa ReLU aktivacijom i Dropout regularizacijom. Trenira se Adam optimizerom, CrossEntropy loss funkcijom, 10 epoha.

```
10 000 → [512, ReLU, Dropout] → [128, ReLU, Dropout] → 2
```

In [ ]:
from sklearn.preprocessing import StandardScaler as SS
scaler_tfidf = SS(with_mean=False)
X_mlp_train_t = torch.FloatTensor(scaler_tfidf.fit_transform(X_tfidf_train).toarray())
X_mlp_test_t  = torch.FloatTensor(scaler_tfidf.transform(X_tfidf_test).toarray())
y_train_t     = torch.LongTensor(y_train)
y_test_t      = torch.LongTensor(y_test)

mlp = MLP(input_size=MAX_TFIDF).to(device)
mlp_hist = train_model(
    mlp, make_loader(X_mlp_train_t, y_train_t, BATCH_SIZE_MLP),
    nn.CrossEntropyLoss(), optim.Adam(mlp.parameters(), lr=LR),
    NUM_EPOCHS, device
)

plt.figure(figsize=(6, 4))
plt.plot(mlp_hist, color='steelblue')
plt.title('MLP — Kretanje loss funkcije')
plt.xlabel('Epoha')
plt.ylabel('Loss')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
y_pred_mlp = predict(mlp, X_mlp_test_t, device)
results['MLP'] = print_and_plot_metrics(y_test, y_pred_mlp, 'MLP', 'plots/cm_mlp.png')

MLP postiže ~99.1% tačnosti i FPR svega 0.35% — znači samo 1 od ~286 ljudskih tekstova biva pogrešno označen kao AI. Ovaj rezultat je posebno važan u kontekstu akademskog integriteta.

## 6. Model 3 — CNN (1D, Word2Vec)

Analogno CNN za slike (vježba 8), ali sa **1D konvolucijama** koje klize duž sekvence riječi umjesto 2D konvolucija koje klize po pikselima. Embedding sloj je inicijalizovan pretreniranim Word2Vec težinama.

```
sekvenca (200 indeksa)
  → Embedding (27215 × 100)         # Word2Vec inicijalizacija
  → Conv1D(128, k=3) + ReLU
  → MaxPool1D(2)
  → Conv1D(64, k=3) + ReLU
  → GlobalMaxPool
  → Dropout → Linear(2)
```

In [ ]:
VOCAB_SIZE = len(word2idx) + 1
X_seq_train_t = torch.LongTensor(X_seq_train)
X_seq_test_t  = torch.LongTensor(X_seq_test)

cnn = TextCNN(VOCAB_SIZE, EMBED_DIM, embedding_matrix).to(device)
print(cnn)

cnn_hist = train_model(
    cnn, make_loader(X_seq_train_t, y_train_t, BATCH_SIZE_CNN),
    nn.CrossEntropyLoss(), optim.Adam(cnn.parameters(), lr=LR),
    NUM_EPOCHS, device
)

plt.figure(figsize=(6, 4))
plt.plot(cnn_hist, color='salmon')
plt.title('CNN — Kretanje loss funkcije')
plt.xlabel('Epoha')
plt.ylabel('Loss')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
y_pred_cnn = predict(cnn, X_seq_test_t, device)
results['CNN'] = print_and_plot_metrics(y_test, y_pred_cnn, 'CNN (1D, Word2Vec)', 'plots/cm_cnn.png')

CNN postiže ~98.9% tačnosti. Za razliku od MLP koji gleda pojedinačne tokene, CNN hvata **lokalne obrasce** (n-grame) kroz konvolucione filtere — što mu daje prednost u razumijevanju stila pisanja.

## 7. K-fold unakrsna validacija (k=5)

Za robusnu procjenu performansi koristimo **5-fold stratifikovanu unakrsnu validaciju**. Skup se dijeli na 5 jednakih dijelova; svaki put jedan dio je test skup a preostalih 4 su trening. Zbog računarskih ograničenja, koristimo podskupove: 15 000 instanci za KNN i MLP, 10 000 za CNN.

In [ ]:
kfold_knn(X_knn_train, y_train, best_k, n_sample=15_000)

In [ ]:
kfold_torch(
    model_fn=lambda: MLP(input_size=MAX_TFIDF),
    X_tensor=X_mlp_train_t, y_tensor=y_train_t,
    device=device, batch_size=BATCH_SIZE_MLP,
    model_name='MLP', n_sample=15_000
)

In [ ]:
kfold_torch(
    model_fn=lambda: TextCNN(VOCAB_SIZE, EMBED_DIM, embedding_matrix),
    X_tensor=X_seq_train_t, y_tensor=y_train_t,
    device=device, batch_size=BATCH_SIZE_CNN,
    model_name='CNN', n_sample=10_000
)

Rezultati K-fold CV su konzistentni sa rezultatima na test skupu — nema znakova prenaučenosti (overfitting-a).

## 8. Poređenje modela i zaključak

In [ ]:
import pandas as pd
df_res = pd.DataFrame(results).T
df_res.columns = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'FPR']
df_res.round(4)

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
x = np.arange(len(metrics))
width = 0.25
colors = ['steelblue', 'salmon', 'mediumseagreen']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for i, (mname, row) in enumerate(df_res.iterrows()):
    ax1.bar(x + i*width, row[metrics].values, width, label=mname, color=colors[i])
ax1.set_xticks(x + width)
ax1.set_xticklabels(metrics)
ax1.set_ylim(0.9, 1.01)
ax1.set_ylabel('Vrijednost')
ax1.set_title('Poređenje modela — test skup')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

bars = ax2.bar(df_res.index, df_res['FPR'], color=colors)
for bar, v in zip(bars, df_res['FPR']):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.001, f'{v:.4f}', ha='center')
ax2.set_title('False Positive Rate\n(ljudski tekst → AI)')
ax2.set_ylabel('FPR')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Zaključak

| Model | Accuracy | F1 | FPR | Obeležja |
|---|---|---|---|---|
| **KNN** | 95.7% | 0.957 | 4.28% | TF-IDF (SVD) + numerička |
| **CNN** | 98.9% | 0.989 | 1.43% | Word2Vec sekvence |
| **MLP** | **99.1%** | **0.991** | **0.35%** | TF-IDF |

**MLP** ostvaruje najbolje rezultate zahvaljujući visokoj dimenzionalnosti TF-IDF reprezentacije koja dobro hvata leksičke karakteristike AI teksta.  
**CNN** je blizak, s prednosti u hvatanju lokalnih stilskih obrazaca kroz konvolucije.  
**KNN** je solidan ali ograničen kletva dimenzionalnosti i visokim FPR-om.

**False positive rate MLP-a od 0.35%** je posebno važan rezultat — u sistemu za detekciju akademske nepoštenosti, svega 1 od ~286 studentskih radova bio bi pogrešno označen kao AI-generisan.